In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"

# Human vs LLM Zero & Few Shot Label Analysis

This part of the notebook compares the human vs llm generated labels for accident text descriptions

In [ ]:
# Data Paths
unfall_table_path = "data/D_Unfall_RData.csv"
unfalltext_table_path = "data/D_Unfalltext_image.csv"
translated_unfalltext_table_path = "data/translated_unfalltext_0_103111.csv"
few_shot_llm_predicted_labels_table_path = "data/few_shot_llm_predicted_labels_unfalltext_0_103111.csv"

In [ ]:
unfall_table = pd.read_csv(unfall_table_path)
unfalltext_table = pd.read_csv(unfalltext_table_path)
translated_unfalltext_table = pd.read_csv(translated_unfalltext_table_path)
few_shot_llm_predicted_labels_table = pd.read_csv(few_shot_llm_predicted_labels_table_path)

In [ ]:
few_shot_llm_predicted_labels_table[few_shot_llm_predicted_labels_table.few_shot_LLM_Labels.notna()].shape

In [ ]:
# Merge the two tables on UN_KEY
unfall_text_table_labels_merged = unfall_table.merge(
    unfalltext_table, on="UN_KEY", how="left").merge(
        few_shot_llm_predicted_labels_table[["UN_KEY", "few_shot_LLM_Labels"]], on="UN_KEY", how="left").merge(
            translated_unfalltext_table[["UN_KEY", "Translated_Text"]], on="UN_KEY", how="left")

# Remove the rows not existing in unfalltext_table (so they don't have accident text descriptions)
unfall_text_table_labels_merged = unfall_text_table_labels_merged[unfall_text_table_labels_merged.UN_KEY.isin(unfalltext_table.UN_KEY)]

# Date columns
unfall_text_table_labels_merged["DateTime"] = pd.to_datetime(unfall_text_table_labels_merged["DateTime"], errors="coerce")
unfall_text_table_labels_merged["Year"] = unfall_text_table_labels_merged["DateTime"].dt.year
unfall_text_table_labels_merged["Month"] = unfall_text_table_labels_merged["DateTime"].dt.month
unfall_text_table_labels_merged["Year_Month"] = unfall_text_table_labels_merged["DateTime"].dt.to_period("M")

unfall_text_table_labels_merged.shape

In [ ]:
# Remove duplicated text list from Sascha

# Read the duplicated text description list
duplicated_texts = pd.read_excel("/home/accidents/data/contributions_wise2425/duplicated_text_description_list.xlsx")
duplicated_texts_to_remove = duplicated_texts[duplicated_texts["filter"]=="yes"]["Text"].tolist()

# Remove the duplicated text descriptions
unfall_text_table_labels_merged = unfall_text_table_labels_merged[~unfall_text_table_labels_merged["Text"].isin(duplicated_texts_to_remove)]
unfall_text_table_labels_merged.shape

In [ ]:
unfall_text_table_labels_merged.head(3)

In [ ]:
unfall_text_table_labels_merged.info()

In [ ]:
unfall_text_table_labels_merged.few_shot_LLM_Labels.value_counts()

In [ ]:
unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels==99].Text

In [ ]:
# Remove row with invalid few_shot_LLM_Labels which contains very short text
unfall_text_table_labels_merged = unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels!=99]

In [ ]:
# Example accident description and labels
idx = 0
print("German Text: ", unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels.notna()].iloc[idx].Text)
print("Translated Text: ", unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels.notna()].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels.notna()].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels.notna()].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Number of Human Labels vs LLM Labels
print("Number of Human Labels: ", unfall_text_table_labels_merged.Unfalltyp.count())
print("Number of Few Shot LLM Labels: ", unfall_text_table_labels_merged.few_shot_LLM_Labels.count())

In [ ]:
# Matched Human Labels and LLM Labels Accident Count
print("Matched Human Labels and Few Shot LLM Label Accident Count: ", unfall_text_table_labels_merged[unfall_text_table_labels_merged.Unfalltyp == unfall_text_table_labels_merged.few_shot_LLM_Labels].shape[0])
print("Ratio of Matched Human Labels and LLM Label Accident Count: ", \
      unfall_text_table_labels_merged[unfall_text_table_labels_merged.Unfalltyp == unfall_text_table_labels_merged.few_shot_LLM_Labels].shape[0] / unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels.notna()].shape[0])

## Comparison Visualizations

In [ ]:
from explorative.conventions import unfalltyp_mapping_eng

In [ ]:
# Convert them to long format for easier plotting
df_long = pd.melt(
    unfall_text_table_labels_merged[unfall_text_table_labels_merged.few_shot_LLM_Labels.notna()], 
    value_vars=['Unfalltyp', 'few_shot_LLM_Labels'], 
    var_name='Source', 
    value_name='Label'
)

# Rename the values in the 'Source' column
df_long['Source'] = df_long['Source'].replace({
    'Unfalltyp': 'Human Labels', 
    'few_shot_LLM_Labels': 'LLM Few-Shot Labels'
})

# Ensure labels are integers for sorting
df_long['Label'] = df_long['Label'].astype(int)

# Calculate the counts for each Source and Label
label_counts = df_long.groupby(['Source', 'Label']).size().reset_index(name='Count')

# Calculate the total counts for each Source
source_totals = df_long['Source'].value_counts().to_dict()

# Add a 'Percentage' column by dividing by the total counts per Source
label_counts['Percentage'] = label_counts.apply(lambda row: (row['Count'] / source_totals[row['Source']]) * 100, axis=1)

palette = {
    'Human Labels': 'tab:orange',
    'LLM Few-Shot Labels': 'tab:blue'
}

# Plot the histogram as side-by-side bars with percentages
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=label_counts.sort_values('Label'),  # Sort by Label to ensure proper order
    x='Label', 
    y='Percentage',
    hue='Source',
    dodge=True,
    palette=palette  # Apply the reversed colors    
)

# Add the percentage text on each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f%%", label_type='edge', fontsize=10)

# Generate combined labels (e.g., "1: Driving accident") for the x-axis
sorted_labels = sorted(unfalltyp_mapping_eng.keys())
combined_labels = [f"{key}: {unfalltyp_mapping_eng[key]}" for key in sorted_labels]

# Set the sorted labels on the x-axis
plt.xticks(ticks=range(len(sorted_labels)), labels=combined_labels, rotation=45, ha='right')
plt.ylabel("Percentage")
plt.legend(title="Label Source")
plt.title("Histogram of Human Labels vs Few Shot LLM Labels (Percentage)")
plt.show()

In [ ]:
llm_labels_per_human_label = unfall_text_table_labels_merged.groupby('Unfalltyp')['few_shot_LLM_Labels'].value_counts().unstack().fillna(0)
llm_labels_per_human_label

In [ ]:
# Show the count of llm labels for each human label category
# Show it as percentage accross the human label category
llm_labels_per_human_label = unfall_text_table_labels_merged.groupby('Unfalltyp')['few_shot_LLM_Labels'].value_counts().unstack().fillna(0)
llm_labels_per_human_label_percentage = (llm_labels_per_human_label.div(llm_labels_per_human_label.sum(axis=1), axis=0) * 100).round(0)
llm_labels_per_human_label_percentage

In [ ]:
# Prepare the data (keeping only integer values for axes)
distr_plot_data = llm_labels_per_human_label_percentage.copy()
categories = [int(x) for x in distr_plot_data.index]  # Only integers
llm_labels = [int(x) for x in distr_plot_data.columns]  # Only integers

# Set up the plot
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_xlim(-0.5, len(categories)-0.5)
ax.set_ylim(-0.5, len(llm_labels)-0.5)

# Create the bubble chart
for i, category in enumerate(categories):
    for j, label in enumerate(llm_labels):
        percentage = distr_plot_data.iloc[i, j]
        if percentage > 0:  # Only plot non-zero percentages
            circle_size = percentage * 100  # Adjust size scaling factor
            ax.scatter(i, j, s=circle_size, color='skyblue', alpha=0.7, edgecolor='black')
            circle_font = 'bold' if percentage >= 20 else 'normal'
            circle_font_size = 20 if percentage >= 20 else 10
            ax.text(i, j, f"{int(percentage)}%", ha='center', va='center', 
                    fontsize=circle_font_size, weight=circle_font)

# Adjust ticks and labels (only integer values)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=20)
ax.set_xlabel("Human Labels", fontsize=20)

ax.set_yticks(range(len(llm_labels)))
ax.set_yticklabels(llm_labels, fontsize=20)
ax.set_ylabel("LLM Few-Shot Labels", fontsize=20)

# Add a separate legend for category descriptions
legend_patches = [mpatches.Patch(color='white', label=f"{int(k)}: {v}") 
                  for k, v in unfalltyp_mapping_eng.items()]
ax.legend(handles=legend_patches, loc='upper left', fontsize=16, title="Label Mappings")

# Add grid for better readability
ax.grid(True, linestyle='--', alpha=0.6)

# Add title
plt.title("LLM Few-Shot Labels vs Human Labels", fontsize=20)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare the data (keeping only integer values for axes)
distr_plot_data = llm_labels_per_human_label_percentage.copy()
distr_plot_data = distr_plot_data[-1:]
categories = [int(x) for x in distr_plot_data.index]  # Only integers
llm_labels = [int(x) for x in distr_plot_data.columns]  # Only integers

# Set up the plot
fig, ax = plt.subplots(figsize=(6, 12))
ax.set_xlim(-0.5, 0.5)
ax.set_ylim(-1, len(llm_labels))

# Create the bubble chart
for i, category in enumerate(categories):
    for j, label in enumerate(llm_labels):
        percentage = distr_plot_data.iloc[i, j]
        if percentage > 0:  # Only plot non-zero percentages
            circle_size = percentage * 100  # Adjust size scaling factor
            ax.scatter(i, j, s=circle_size, color='skyblue', alpha=0.7, edgecolor='black')
            circle_font = 'bold' if percentage >= 20 else 'normal'
            circle_font_size = 20 if percentage >= 20 else 10
            ax.text(i, j, f"{int(percentage)}%", ha='center', va='center', 
                    fontsize=circle_font_size, weight=circle_font)

# Adjust ticks and labels (only integer values)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=20)
ax.set_xlabel("Human Labels", fontsize=20)

ax.set_yticks(range(len(llm_labels)))
ax.set_yticklabels(llm_labels, fontsize=20)
ax.set_ylabel("LLM Few-Shot Labels", fontsize=20)

# Add a separate legend for category descriptions
legend_patches = [mpatches.Patch(color='white', label=f"{int(k)}: {v}") 
                  for k, v in unfalltyp_mapping_eng.items()]
# ax.legend(handles=legend_patches, loc='upper left', fontsize=16, title="Label Mappings")

# Add grid for better readability
ax.grid(True, linestyle='--', alpha=0.6)

# Add title
plt.title("LLM Few-Shot Labels vs Human Labels", fontsize=20)
plt.tight_layout()
plt.show()


## Case 1) Human Label 7 & LLM Label 5 Cases

In [ ]:
# 35K cases (34% of all accidents)
print("Count of cases: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].shape[0])
print("Percentage of this case over all accidents: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].shape[0] / unfall_text_table_labels_merged.shape[0])

In [ ]:
# Some examples for the accidents with human label 7 and LLM label 5
idx = 47 # 47, 42
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Some examples for the accidents with human label 5 and LLM label 5
idx = 42 # 42, 40
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 5) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 5) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 5) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 5) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Strings that can be contained in an accident description for a parking accident
parken_match_str_regex = r"garage|park|stell|partk|pakten|pakrt"

In [ ]:
# Search for the "park" keyword in the translated text
# 99% cases contain the "park" keyword
park_keyword_count = unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5) & 
                                (unfall_text_table_labels_merged.Text.str.contains(parken_match_str_regex, case=False))].shape[0]
print("Number of cases with 'park' keyword: ", park_keyword_count)
print("Percentage of cases with 'park' keyword inside the Human Label 7 LLM Label 5 intersection: ", 
      park_keyword_count / unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)].shape[0])

In [ ]:
# Some examples that DO NOT contain PARK and human label 7 and LLM label 5
idx = 1
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5) & 
                                ~(unfall_text_table_labels_merged.Text.str.contains(parken_match_str_regex, case=False))].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5) & 
                                ~(unfall_text_table_labels_merged.Text.str.contains(parken_match_str_regex, case=False))].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5) & 
                                ~(unfall_text_table_labels_merged.Text.str.contains(parken_match_str_regex, case=False))].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5) & 
                                ~(unfall_text_table_labels_merged.Text.str.contains(parken_match_str_regex, case=False))].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# If we create category 8 for the case Human Label 7 and LLM Label 5
categories_with_additonal_class_8 = unfall_text_table_labels_merged.apply(
    lambda x: 8 if (x.Unfalltyp == 7) & (x.few_shot_LLM_Labels == 5) else x.Unfalltyp, axis=1)

In [ ]:
unfalltyp_mapping_eng_with_additional_class_8 = unfalltyp_mapping_eng.copy()
unfalltyp_mapping_eng_with_additional_class_8[8] = "Damaged Parked Vehicle"
unfalltyp_mapping_eng_with_additional_class_8

In [ ]:
plt.figure(figsize=(10, 5))

# Calculate the percentage for each category
value_counts = categories_with_additonal_class_8.value_counts()
percentages = value_counts / value_counts.sum() * 100

# Plot the percentages
ax = sns.barplot(
    x=percentages.index,
    y=percentages.values
)

# Add the percentage text on each bar
ax.bar_label(ax.containers[0], fmt="%.0f%%", label_type='edge', fontsize=10)

# Generate combined labels (e.g., "1: Driving accident") for the x-axis
sorted_labels = categories_with_additonal_class_8.sort_values().unique()
combined_labels = [f"{key}: {unfalltyp_mapping_eng_with_additional_class_8[key]}" for key in sorted_labels]

plt.xticks(ticks=range(len(sorted_labels)), labels=combined_labels, rotation=45, ha='right')
plt.xlabel("LLM Labels with Additional Class 8")
plt.ylabel("Percentage")
plt.title("Histogram of Few Shot LLM Labels with Additional Class 8")
plt.show()

## Case 3) Human Label 6 & LLM Label 3 or 2

In [ ]:
# 2K
unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 6) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels.isin([2, 3]))].shape[0]

In [ ]:
# Some examples for the accidents with human label 6 and LLM label 3
idx = 8
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 6) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 3)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 6) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 3)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 6) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 3)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 6) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == 3)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Generally accidents occur longitudinal but just after turning

## Case 2) Label 2 and 3 Confusions

In [ ]:
print("Label 2 and 3 confusion discrepency ratio over all accidents: ", \
      (unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 2) & (unfall_text_table_labels_merged.few_shot_LLM_Labels == 3)].shape[0] + \
       unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 3) & (unfall_text_table_labels_merged.few_shot_LLM_Labels == 2)].shape[0]) \
        / unfall_text_table_labels_merged.shape[0])
                                                       

In [ ]:
# Matched Case Examples
idx=2
class_no=2
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Human label 2 and LLM label 3 Case
idx=4
human_class_no=2
llm_class_no=3
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Human label 3 and LLM label 2 Case
idx=7
human_class_no=3
llm_class_no=2
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# LLM has hard time differentiating between class 2 and 3. Human label is more accurate in this case.

## Other Discrepancies (Small friction of all accidents)

In [ ]:
# Human label 7 and LLM label 1 Case
# COMMENT: Accidents are mostly about alone accidents, some of them are actually class 1 but some others contain techinical reasons/failures which are class 7
idx=1
human_class_no=7
llm_class_no=1
print("Volume of cases: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].shape[0])
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Human label 7 and LLM label 2 Case
# COMMENT: Descriptions contain turning but most of the time not with a car, so it should be class 7?
idx=43
human_class_no=7
llm_class_no=2
print("Volume of cases: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].shape[0])
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Human label 7 and LLM label 3 Case
# COMMENT: No pattern
idx=1
human_class_no=7
llm_class_no=3
print("Volume of cases: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].shape[0])
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Human label 7 and LLM label 4 Case
# COMMENT: Involves pedestrians and some bikes sometimes, but no general pattern
idx=123
human_class_no=7
llm_class_no=4
print("Volume of cases: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].shape[0])
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)

In [ ]:
# Strings that can be contained in an accident description for a pedestriand and bicycle accident
match_str_regex = r"^(?=.*(?:fahrrad|rad|radfahrer))(?=.*(?:fuß|fußgänger)).+$"
# Search keywords
match_keyword_count = unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                (unfall_text_table_labels_merged.few_shot_LLM_Labels == 4) & 
                                (unfall_text_table_labels_merged.Text.str.contains(match_str_regex, case=False))].shape[0]
print("Number of cases with keywords: ", match_keyword_count)
print("Percentage of cases with keyword inside the Human Label 7 LLM Label 4 intersection: ", 
      match_keyword_count / unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == 7) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == 4)].shape[0])

In [ ]:
# Human label 7 and LLM label 6 Case
# COMMENT: No pattern
idx=9
human_class_no=7
llm_class_no=6
print("Volume of cases: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].shape[0])
print("German Text: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Text)
print("Translation: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) & 
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Translated_Text)
print("Human Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                       (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].Unfalltyp)
print("Few Shot LLM Label: ", unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp == human_class_no) &
                                                              (unfall_text_table_labels_merged.few_shot_LLM_Labels == llm_class_no)].iloc[idx].few_shot_LLM_Labels)